In [1]:
from pathlib import Path
from persona import generate_population_descriptions
from agents import PersonaAgent
from tqdm import tqdm

CONFIG_PATH = Path("configs/ATLAS")
DATA_PATH = CONFIG_PATH / "data"

descriptions, population_sample = generate_population_descriptions(
    config_folder=CONFIG_PATH,
    data_path=DATA_PATH,
    n_sample=1000
)

In [2]:
import json
from pathlib import Path
from agents import RoleplayingAgent

persona_generator = PersonaAgent()
roleplay_agent = RoleplayingAgent()

n = 100
t = 10  # flush to cache every t iterations

cache_dir = Path("run/cache")
cache_dir.mkdir(parents=True, exist_ok=True)

buffer = []
chunk_idx = 0
skipped = []

for i, (description, population_package) in enumerate(zip(descriptions[:n], population_sample.iloc[:n, :].iterrows())):
    agent_idx, agent_info = population_package

    try:
        persona = persona_generator.run(demographic_description=description)
        plan = roleplay_agent.run(persona)
    except Exception as e:
        print(f"[skip] agent {agent_idx} failed at iteration {i}: {type(e).__name__}: {e}")
        skipped.append({"agent_idx": agent_idx, "iteration": i, "error": str(e)})
        continue

    record = {
        "agent_idx": agent_idx,
        "SERIALNO": agent_info["SERIALNO"],
        "PUMA": agent_info["PUMA"],
        "self_introduction": plan.self_introduction,
        "travel_plans_summary": plan.travel_plans_summary,
        "itinerary": {
            "locations": plan.locations,
            "location_context": plan.location_context,
            "departure_times": plan.departure_times,
        },
    }
    buffer.append(record)

    if (i + 1) % t == 0:
        chunk_path = cache_dir / f"chunk_{chunk_idx:04d}.jsonl"
        with open(chunk_path, "w") as f:
            for rec in buffer:
                f.write(json.dumps(rec) + "\n")
        print(f"[cache] wrote {len(buffer)} records → {chunk_path}")
        buffer.clear()
        chunk_idx += 1

# flush any remaining records that didn't fill a full chunk
if buffer:
    chunk_path = cache_dir / f"chunk_{chunk_idx:04d}.jsonl"
    with open(chunk_path, "w") as f:
        for rec in buffer:
            f.write(json.dumps(rec) + "\n")
    print(f"[cache] wrote {len(buffer)} records → {chunk_path}")

# concatenate all cache chunks into final output
output_path = Path("run/output.jsonl")
with open(output_path, "w") as out:
    for chunk_path in sorted(cache_dir.glob("chunk_*.jsonl")):
        out.write(chunk_path.read_text())

print(f"[done] {n - len(skipped)}/{n} agents written → {output_path}")
if skipped:
    print(f"[skipped] {len(skipped)} agents: {[s['agent_idx'] for s in skipped]}")


[skip] agent 2 failed at iteration 2: InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    1 validation error for DailyPlan
  Invalid JSON: EOF while parsing an object at line 20 column 15 [type=json_invalid, input_value='{\n  "self_introduction"...    \n\n               ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid
</exception>
<completion>
    ChatCompletion(id='chatcmpl-924', choices=[Choice(finish_reason=None, index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "self_introduction": "I\'m Ethan Martin, a 30-year-old waiter who\'s spent the better part of a decade working at the local family diner. I\'ve built up a regular routine - 40 hours a week means I\'m never short on stories about my eccentric customers or exhausting evenings. But when I do get some free time, you can find me hitting the neighborhood courts with friends for some pickup basketball or sneaking away to the park 